# 支付阶段整理：从原始 CSV 独立复现

你已经完成第六份 Notebook 的探索和原始记录复核。本次把步骤串成可重复运行的函数流程。

请先保存第六份 Notebook，再在此处选择内核并逐格运行。本文件只读取原始数据并在内存中计算；终端导出命令见末尾。

课堂基线：103,886 条支付记录；99,441 笔主表订单；98,665 笔可核对、98,089 笔完全一致、576 笔有差额、776 笔无法核对。每次重新运行都从原始 CSV 开始，无需前面 Notebook 的变量或商品阶段快照。

In [3]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'
src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from items_cleaning import (
    clean_order_items, aggregate_order_items, reconcile_item_totals
)
from payments_cleaning import (
    PAYMENT_FLAG_COLS, clean_order_payments, aggregate_order_payments,
    join_orders_payments, reconcile_payment_totals, reconcile_order_amounts,
    amount_difference_distribution, summarize_payment_metrics, recheck_raw_order
)

def read_raw(filename):
    return pd.read_csv(
        data_dir / filename, dtype=str, encoding='utf-8-sig',
        keep_default_na=False, na_values=['']
    )

raw_payments = read_raw('olist_order_payments_dataset.csv')
raw_orders = read_raw('olist_orders_dataset.csv')
raw_items = read_raw('olist_order_items_dataset.csv')
print('支付、订单、商品明细行数：', len(raw_payments), len(raw_orders), len(raw_items))

支付、订单、商品明细行数： 103886 99441 112650


## 1. 支付记录清洗与订单汇总

clean_order_payments 输入原始字符串表，返回包含全部原字段、整数分和核查标记的新表。它保留零金额、未定义支付方式、零期数；输入主键或金额无法安全处理时会报错。

aggregate_order_payments 把支付记录汇总为订单。期数不参与金额相乘，订单级 any 标记表示该订单至少有一条记录命中对应规则。

In [4]:
payments_clean = clean_order_payments(raw_payments)
order_payment_summary = aggregate_order_payments(payments_clean)

display(payments_clean[
    ['payment_sequential', 'payment_installments', 'payment_value_cents']
].dtypes)
display(payments_clean[PAYMENT_FLAG_COLS + ['flag_payment_needs_review']].sum())
print('原始支付表行数：', len(raw_payments))
print('支付工作表行数：', len(payments_clean))
print('订单汇总行数：', len(order_payment_summary))
order_payment_summary.head()

payment_sequential      Int64
payment_installments    Int64
payment_value_cents     int64
dtype: object

flag_zero_payment                 9
flag_payment_type_not_defined     3
flag_zero_installments            2
flag_payment_needs_review        11
dtype: int64

原始支付表行数： 103886
支付工作表行数： 103886
订单汇总行数： 99440


,order_id,payment_record_count,payment_amount_cents,has_zero_payment_record,has_undefined_payment_type,has_zero_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,7219,False,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,25983,False,False,False
2,000229ec398224ef6ca0657da4fc703e,1,21687,False,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,2578,False,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,21804,False,False,False


## 2. 订单覆盖与同来源控制总额

保留订单主表全部字段，反向检查支付订单能否在主表找到，再左连接。没有支付记录的订单，其金额、记录数与支付特征保持未知。

支付明细、订单汇总、关联后的记录数和金额应一致。这与后面的跨来源金额差异是两种检查。

In [5]:
orders_with_payments = join_orders_payments(raw_orders, order_payment_summary)
payment_controls = reconcile_payment_totals(
    payments_clean, order_payment_summary, orders_with_payments
)

display(payment_controls)
display(orders_with_payments['payments_match'].value_counts())
orders_with_payments.loc[
    orders_with_payments['flag_missing_payments'],
    ['order_id', 'order_status', 'order_purchase_timestamp',
     'payment_record_count', 'payment_amount_cents']
]

,明细层,订单汇总层,关联后,汇总差额,关联差额
支付记录数,103886,103886,103886,0,0
支付金额（分）,1600887212,1600887212,1600887212,0,0


payments_match
both          99440
left_only         1
right_only        0
Name: count, dtype: int64

,order_id,order_status,order_purchase_timestamp,payment_record_count,payment_amount_cents
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,<NA>,<NA>


## 3. 与商品金额、运费逐单核对

复用商品阶段函数，从原始明细生成订单汇总。差额定义为支付金额减去商品金额和运费。

三项金额都非空才可核对；完全一致的标准为差额等于零。无法核对时，payment_difference_cents 和 flag_amount_mismatch 都保持未知，amount_reconcile_status 明确记录三种状态。

In [6]:
items_clean = clean_order_items(raw_items)
order_item_summary = aggregate_order_items(items_clean)
orders_amount_check = reconcile_order_amounts(
    orders_with_payments, order_item_summary
)
item_controls = reconcile_item_totals(
    items_clean, order_item_summary, orders_amount_check
)
metrics = summarize_payment_metrics(
    payments_clean, order_payment_summary, orders_amount_check
)

display(item_controls)
pd.Series({
    '总订单数': metrics['all_order_rows'],
    '可核对订单数': metrics['amount_comparable_orders'],
    '无法核对订单数': metrics['amount_uncomparable_orders'],
    '金额完全一致订单数': metrics['amount_equal_orders'],
    '存在金额差额订单数': metrics['amount_mismatch_orders']
})

,明细层,订单汇总层,关联后,汇总差额,关联差额
商品明细数量,112650,112650,112650,0,0
商品金额（分）,1359164370,1359164370,1359164370,0,0
运费（分）,225190954,225190954,225190954,0,0


总订单数         99441
可核对订单数       98665
无法核对订单数        776
金额完全一致订单数    98089
存在金额差额订单数      576
dtype: int64

In [7]:
display(pd.Series({
    '零金额支付记录数': metrics['zero_payment_rows'],
    '含零金额记录的订单数': metrics['orders_with_zero_payment_record'],
    '订单支付总额为零的订单数': metrics['zero_payment_total_orders'],
    '支付方式未定义记录数': metrics['undefined_type_rows'],
    '零期数记录数': metrics['zero_installment_rows'],
    '上述标记并集的记录数': metrics['flagged_payment_rows']
}))

# 含零金额记录和整笔订单支付总额为零，需要分别判断。
order_payment_summary.loc[
    order_payment_summary['has_zero_payment_record'],
    ['order_id', 'payment_record_count', 'payment_amount_cents']
]

零金额支付记录数         9
含零金额记录的订单数       8
订单支付总额为零的订单数     3
支付方式未定义记录数       3
零期数记录数           2
上述标记并集的记录数      11
dtype: int64

,order_id,payment_record_count,payment_amount_cents
251,00b1cb0320190ca0daa2c88b35206009,1,0
27157,45ed6e85398a87c253db47c2d9f48216,3,7114
27282,4637ca194b6387e2d538dc89b124b0ee,1,0
42262,6ccb433e00daae1283ccc956189c82ae,4,12204
54093,8bcbe01d44d147f901cd3192671144db,4,7416
69070,b23878b3e8eb4d25a158f57d96331b18,4,17157
78007,c8c528189310eaa44a745b8d9d26908b,1,0
97261,fa65dad1b0e818e3ccc5cb0e39231352,29,45799


## 4. 差额分布与原始样本复核

方向与大小是两个独立分类。1 分、2～99 分、至少 100 分用于描述分布，仍保留所有非零差额。

优先复核绝对差额最大的样本；原始商品金额、运费和支付额从字符串重新换算，不能直接拿汇总结果充当独立复核。

In [8]:
display(amount_difference_distribution(orders_amount_check))

difference_orders = orders_amount_check.loc[
    orders_amount_check['flag_amount_mismatch'].fillna(False)
].copy()
top_difference_orders = difference_orders.nlargest(5, 'abs_difference_cents')
display(top_difference_orders[
    ['order_id', 'order_status', 'goods_amount_cents',
     'freight_amount_cents', 'payment_amount_cents', 'payment_difference_cents']
])

if not top_difference_orders.empty:
    check_order_id = top_difference_orders.iloc[0]['order_id']
    print('本次复核订单：', check_order_id)
    case_recheck = recheck_raw_order(
        raw_items, raw_payments, orders_amount_check, check_order_id
    )
    display(raw_items.loc[
        raw_items['order_id'].eq(check_order_id),
        ['order_id', 'order_item_id', 'price', 'freight_value']
    ])
    display(raw_payments.loc[
        raw_payments['order_id'].eq(check_order_id),
        ['order_id', 'payment_sequential', 'payment_type',
         'payment_installments', 'payment_value']
    ])
    display(case_recheck)

,分类维度,分组,订单数
0,方向,支付额较大,367
1,方向,支付额较小,209
2,大小,绝对差额1分,273
3,大小,绝对差额2至99分,53
4,大小,绝对差额100分及以上,250


,order_id,order_status,goods_amount_cents,freight_amount_cents,payment_amount_cents,payment_difference_cents
11791,ce6d150fb29ada17d2082f4847107665,delivered,129900,10466,158647,18281
48686,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,17919,10872,40692,11901
70865,70b742795bc441e94a44a084b6d9ce7a,delivered,26999,19694,57882,11189
33150,996c7e73600ad3723e8627ab7bef81e4,delivered,55990,2800,66443,7653
52985,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,16788,4527,27484,6169


本次复核订单： ce6d150fb29ada17d2082f4847107665


,order_id,order_item_id,price,freight_value
90875,ce6d150fb29ada17d2082f4847107665,1,1299.00,104.66


,order_id,payment_sequential,payment_type,payment_installments,payment_value
359,ce6d150fb29ada17d2082f4847107665,1,credit_card,10,1586.47


,原始明细重算,订单级结果,差值
商品金额（分）,129900,129900,0
运费（分）,10466,10466,0
支付金额（分）,158647,158647,0
金额差额（分）,18281,18281,0


## 5. 对照结果与导出

预期：两个控制总额表的差额列全部为零；三种金额核对状态分别为 98,089、576、776 笔。最大绝对差额样本重算得到 18,281 分，与订单级结果一致。

这笔样本的原始文件已有相同差异，本次汇总和关联没有新增其差额；业务原因仍待核查。只完成一笔样本复核，不能据此替其他订单确定原因。

在项目目录终端运行 python .\src\04_clean_payments.py，可导出 data/processed/payments_v1 与 reports/payments_v1 的数据和阶段报告。它同样从原始 CSV 独立执行。

CSV 重读时须恢复可空整数、布尔及日期类型。当前为全量历史、全部状态的核对，正式分析窗口和敏感性分析等仍在后续阶段。

## 我的理解

1. payments_clean = clean_order_payments(raw_payments) 的输入、返回值各是什么？返回表与原表是什么关系？

我的回答：输入的是原始的支付表，返回的是保留原字段、增加整数分和核查标记的整张新表；函数内部使用 .copy()，保护原始支付表。

2. 同来源控制总额不一致时，为什么应停止流程？跨来源存在 576 笔金额差额时，为什么要保留并输出清单？

我的回答：同来源控制总额不一致，说明处理过程可能存在遗漏、重复或计算错误，应停止并核查。跨来源的金额差异可能已存在于原始数据中，应保留原值并输出清单，继续调查原因。

3. flag_amount_mismatch 为缺失与为 False 分别表示什么？为什么统计金额一致率时，要先明确可核对样本？

我的回答：缺失表示数据不足、无法判断；False 表示已完成核对且金额一致。统计一致率时，应使用可核对订单作为分母，避免将未知情况当成金额不一致。

本次记录：

独立复现完成后，下一课进入评价表：一行代表什么、review_id 是否能直接作为主键，以及一单多评如何处理。